In [ ]:
import torch 
import math
import torch.nn as nn
import torch.nn.functional as F 
import matplotlib.pyplot as plt
import numpy as np
import requests
from dataclasses import dataclass
from jaxtyping import Float, Int
from torch.optim import Adam 

In [7]:
@dataclass
class Config:
    d_model: int 
    d_vocab: int
    d_hidden: int #bigger than d_model (around 4x usually)
    d_embedding: int
   
#Config class is container for Transformer dimensions

## Getting Data

In [8]:
url = "https://www.gutenberg.org/files/64317/64317-0.txt"
response = requests.get(url)
text = response.text 

start_indicate = "START OF THE PROJECT GUTENBERG EBOOK"
end_indicate = "END OF THE PROJECT GUTENBERG EBOOK"

start_idx = text.find(start_indicate)
end_idx = text.find(end_indicate)
content_area = text[start_idx:end_idx].split("\n", 1)[1]

print(content_area[:500000])





                           The Great Gatsby
                                  by
                          F. Scott Fitzgerald


                           Table of Contents

I
II
III
IV
V
VI
VII
VIII
IX


                              Once again
                                  to
                                 Zelda


  Then wear the gold hat, if that will move her;
  If you can bounce high, bounce for her too,
  Till she cry “Lover, gold-hatted, high-bouncing lover,
  I must have you!”

  Thomas Parke d’Invilliers


                                  I

In my younger and more vulnerable years my father gave me some advice
that I’ve been turning over in my mind ever since.

“Whenever you feel like criticizing anyone,” he told me, “just
remember that all the people in this world haven’t had the advantages
that you’ve had.”

He didn’t say any more, but we’ve always been unusually communicative
in a reserved way, and I understood that he meant a great deal more
than that. In conse

## Tokenize the Text

In [ ]:
class Tokenizer:
    def __init__(self, text):
        #process the data
        cleaned = self.clean_text(text)
        self.chars = sorted(set(cleaned))

        self.vocab_size = len(self.chars)
        self.d_vocab = self.vocab_size

        #get vocabulary, store to self
        self.encode = {}
        self.decode = {}    #decode and encode dictionaries

        for i, char in enumerate(self.chars): 
            self.encode[char] = i
            self.decode[i]= char

    def clean_text(self, text : str) -> str:
        #Cleaning the text
        text = text.lower()
        cleaned = []
        for char in text: 
            if char.isalpha() or char in " .?!": #Keep only valid characters
                cleaned.append(char)
        cleaned_text = "".join(cleaned)
        return cleaned_text

        
    def tokenize(self, text):
        #encode w vocab map 
        cleaned = self.clean_text(text)
        tokens = []

        for char in cleaned: 
            if char in self.encode:
                tokens.append(self.encode[char])    
       
        print(f"Encoded output: {tokens} \n")
        return tokens 
    
    def detokenize(self, tokens):
        word_list = []
        for id in tokens: 
            if id in self.decode:
                word_list.append(self.decode[id])
        
        detokenized = "".join(word_list)

        print(f"Decoded output: {detokenized}")

        return detokenized


## Multi-Layer Perceptron

$$
    \texttt{MLP}(\mathbf{X}) = W_d \cdot \sigma_{\texttt{ReLU}} (W_u \cdot x + b_u) + b_d, \qquad \texttt{MLP} : \mathbb{R}^{d_m} \to \mathbb{R}^{d_m}
$$

In [ ]:
class MLP(nn.Module):
    def __init__(self, config: Config):
        super(MLP, self).__init__()
        self.linear_up = nn.Linear(config.d_model, config.d_hidden)
        self.linear_down = nn.Linear(config.d_hidden, config.d_model)

    def forward(self, x:Float[torch.Tensor, "* d_model"]) -> Float [torch.Tensor, "* d_model"]:
        x = self.linear_up(x)
        x = torch.relu(x)
        x = self.linear_down(x)
        return x 

## Attention Head

### Weight Matrix
$$
    \mathbf{W}_{QK} := \mathbf{W}_{Q} \cdot \mathbf{W}_{K}^T, \qquad \mathbf{W}_{Q}, \mathbf{W}_{K} \in \mathbb{R}^{d_m \times d_n}
$$

### Masking Matrix (M)
$$
    M_{i,j} = 
    \begin{cases}
        0 &j \geq i \\
        -\infty &j < i>
    \end{cases}
$$

### Forward Pass
$$A(\mathbf{X}) = \sigma_{\text{softmax}} (\mathbf{X} \; \mathbf{W}_{QK} \; \mathbf{X}^\text{T} + \mathbf{M}) \; \mathbf{X} \; \mathbf{W}_{OV}$$

In [ ]:
class AttentionHead(nn.Module):
    def __init__(self, config: Config):
        super().__init__()
        self.d_model = config.d_model
        self.d_hidden = config.d_hidden

        self.W_q = nn.Linear(self.d_model, self.d_hidden)
        self.W_k = nn.Linear(self.d_model, self.d_hidden)
        self.W_ov = nn.Linear(self.d_model, self.d_hidden)
        self.W_o = nn.Linear(self.d_hidden, self.d_model) # projects back to d_model 
        

    def forward(self, x: Float[torch.Tensor, "n_ctx d_model"]) -> Float[torch.Tensor, "n ctx d_model"]:
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_ov(x)

        #Masking Matrix M
        n_ctx = x.shape[-2]
        M = torch.triu(
            torch.full((n_ctx, n_ctx), float("-inf"), device=x.device),
            diagonal = 1
        )
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_model)
        attn_weights = F.softmax(scores + M, dim = -1)
        output = self.W_o(torch.matmul(attn_weights, V))
    
        return output


## Transformer Block
$$
    \text{TB}(X) = X + A(X) + \text{MLP}(X), \qquad \text{TB}: \mathbb{R}^{n_c \times d_m} \to \mathbb{R}^{n_c \times d_m}
$$

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, config: Config):
        super().__init__()
        self.config = config
        self.A = AttentionHead(config)
        self.mlp = MLP(config)
    
    def forward(self, x: Float[torch.Tensor, "n_ctx d_model"]) -> Float[torch.Tensor, "n_ctx d_model"]:
       x = x + self.A(x)
       x = x + self.mlp(x)
       return x

In [ ]:
class Transformer(nn.Module):
    def __init__(self, config: Config, tokenizer : Tokenizer, max_seq_length, num_blocks):
        super().__init__()
        self.tokenizer = tokenizer
        self.config = config
        self.token_embedding = nn.Embedding(config.d_vocab, config.d_model) 
        self.pos_embedding = nn.Parameter(torch.randn(1, max_seq_length, config.d_model))
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(num_blocks)])
        self.ln = nn.LayerNorm(config.d_model) #LayerNorm stabilizes + improves performance
        self.unembedding = nn.Linear(config.d_model, config.d_vocab)
    def forward(self, x):
        seq_length = x.size(1)
        x = self.token_embedding(x) + self.pos_embedding[:, :seq_length, :]
        for block in self.blocks: #Iterate through ModuleList in forward 
            x = block(x)
        x = self.ln(x)
        logits = self.unembedding(x)
        return logits
    
    def generate(self, text: str, max_length:int, temperature: float = 0.3 )-> str:
        self.eval()
        tokens = self.tokenizer.tokenize(text)
        tokens = torch.tensor(tokens).unsqueeze(0)

        with torch.no_grad():
            for i in range(max_length):
                tokens_short = tokens[:, -self.pos_embedding.shape[1]:] #Ensuring sequence never longer than positional embedding
                logits = self.forward(tokens_short) #Run transformer token sequence 
                next_logits = logits[0, -1, :]
                probabilities = F.softmax(next_logits / temperature, dim = -1)
                new_token = torch.multinomial(probabilities, num_samples = 1)
                tokens = torch.cat([tokens, new_token.unsqueeze(0)], dim= -1) #Appends new token onto end of sequence
        detokenized_txt = self.tokenizer.detokenize(tokens[0].tolist())

        return detokenized_txt
        

## Training Loop

In [ ]:
def training_loop(samples, batch_size, model, n_epochs):
    tokens = torch.tensor(tokenizer.tokenize(content_area))

    #80/20 train test split
    split = int(len(tokens) * 0.8)
    train_tokens = tokens[:split]
    test_tokens = tokens[split:]

    x_train = train_tokens[:-1]
    y_train = train_tokens[1:]

    x_test = test_tokens[:-1]
    y_test = test_tokens[1:]

    seq_length = batch_size

    loss_function = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr = 1e-4)

    train_losses = []
    validation_losses = []

    print("Beginning training")
    model.train()

    #Training losses
    for epoch in range(n_epochs):
        epoch_train_losses = []
        for i in range(0,len(x_train) - seq_length, seq_length):
            x_batch = x_train[i:i+seq_length].unsqueeze(0)
            y_batch = y_train[i:i+seq_length].unsqueeze(0)

            logits = model(x_batch)
            train_loss = loss_function(logits.reshape(-1, config.d_vocab), y_batch.reshape(-1))

            optimizer.zero_grad()
            train_loss.backward()
            optimizer.step()
            epoch_train_losses.append(train_loss.item())

        avg_train_loss = sum(epoch_train_losses) / len(epoch_train_losses) 


        #Validation (uses test splits)
        model.eval()
    
        with torch.no_grad():
            epoch_validation_losses = []
            for i in range(0, len(x_test) - seq_length, seq_length):
                x_batch = x_test[i:i+seq_length].unsqueeze(0)
                y_batch = y_test[i:i+seq_length].unsqueeze(0)

                logits = model(x_batch)
                val_loss = loss_function(logits.reshape(-1, config.d_vocab), y_batch.reshape(-1))
                epoch_validation_losses.append(val_loss.item())
        
        avg_val_loss = sum(epoch_validation_losses) / len(epoch_validation_losses)
        print(f"Epoch {epoch+1} | Train Loss: {train_loss.item():.4f} | Val Loss: {avg_val_loss:.4f}")
        
        train_losses.append(avg_train_loss)
        validation_losses.append(avg_val_loss)

        model.train()

    #Printing Training/Validatiion Loss plot
    print(f"Final Average Training Loss: {avg_train_loss:.4f} | Final Average Validation Loss {avg_val_loss:.4f}")
    epochs = range(1, len(validation_losses) +1)
    plt.figure(figsize = (8,6))
    plt.plot(epochs, train_losses, label = "Training Loss", marker="o", color = "magenta")
    plt.plot(epochs, validation_losses, label = "Validation Losses", marker = "o", color = "purple")
    plt.xlabel("Epoch")
    plt.ylabel("Cross-Entropy Loss")
    plt.title("Training and Validation Loss")
    plt.legend()
    plt.grid(True)
    plt.show()


### Implementing Training

In [ ]:
tokenizer = Tokenizer(content_area)

config = Config(d_model=128, d_vocab=tokenizer.vocab_size, d_hidden=512, d_embedding=128)
model_test = Transformer(config, tokenizer=tokenizer, max_seq_length=128, num_blocks=4)

#Testing untrained model 
print("UNTRAINED model results:")
print(model_test.generate("the green light", max_length = 300))

#Train the model
training_loop(samples=content_area, batch_size=128, model=model_test, n_epochs=10)

#Results of trained model
print("TRAINED MODEL results:")
print(model_test.generate("the green light", max_length = 300))

SyntaxError: expected argument value expression (4188143680.py, line 3)